# CineRec — Análisis Exploratorio de Datos (EDA)

Este notebook analiza el dataset **MovieLens 1M** utilizado en el sistema de recomendación CineRec.  
El objetivo es entender la distribución de los datos antes de entrenar los modelos.

**Dataset:**
- 3.883 películas
- 6.040 usuarios
- 1.000.209 ratings (escala 1-5)

## 1. Importaciones y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt

from pathlib import Path

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

BASE_DIR = Path('..') / 'data' / 'processed'

movies  = pd.read_csv(BASE_DIR / 'movies_clean.csv')
users   = pd.read_csv(BASE_DIR / 'users_clean.csv')
ratings = pd.read_csv(BASE_DIR / 'ratings_clean.csv')

# Parse genres
movies['genres_parsed'] = movies['genres'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

print(f'Movies:  {len(movies):,}')
print(f'Users:   {len(users):,}')
print(f'Ratings: {len(ratings):,}')

## 2. Vista general de los datos

In [ ]:
print('=== MOVIES ===')
display(movies.head())
print('\n=== USERS ===')
display(users.head())
print('\n=== RATINGS ===')
display(ratings.head())

In [ ]:
print('=== INFORMACIÓN DE LOS DATASETS ===')
print('\nMovies:')
print(movies.dtypes)
print(f'Valores nulos: {movies.isnull().sum().sum()}')
print('\nUsers:')
print(users.dtypes)
print(f'Valores nulos: {users.isnull().sum().sum()}')
print('\nRatings:')
print(ratings.dtypes)
print(f'Valores nulos: {ratings.isnull().sum().sum()}')

## 3. Análisis de Ratings

In [ ]:
print('=== ESTADÍSTICAS DE RATINGS ===')
display(ratings['rating'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución de ratings
rating_counts = ratings['rating'].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de Ratings', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Número de valoraciones')
for i, (idx, val) in enumerate(rating_counts.items()):
    axes[0].text(idx, val + 5000, f'{val:,}', ha='center', fontsize=9)

# Ratings por usuario
ratings_per_user = ratings.groupby('userId').size()
axes[1].hist(ratings_per_user, bins=50, color='coral', edgecolor='white')
axes[1].set_title('Distribución de Ratings por Usuario', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Número de ratings')
axes[1].set_ylabel('Número de usuarios')
axes[1].axvline(ratings_per_user.mean(), color='red', linestyle='--', label=f'Media: {ratings_per_user.mean():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Rating medio: {ratings["rating"].mean():.2f}')
print(f'Media de ratings por usuario: {ratings_per_user.mean():.0f}')
print(f'Mínimo: {ratings_per_user.min()} | Máximo: {ratings_per_user.max()}')

In [ ]:
# Ratings por película
ratings_per_movie = ratings.groupby('movieId').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ratings_per_movie, bins=50, color='mediumpurple', edgecolor='white')
axes[0].set_title('Distribución de Ratings por Película', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Número de ratings')
axes[0].set_ylabel('Número de películas')
axes[0].axvline(ratings_per_movie.mean(), color='red', linestyle='--', label=f'Media: {ratings_per_movie.mean():.0f}')
axes[0].legend()

# Top 10 películas más valoradas
top_movies = (
    ratings.groupby('movieId').size()
    .reset_index(name='count')
    .merge(movies[['movieId', 'title']], on='movieId')
    .sort_values('count', ascending=False)
    .head(10)
)
axes[1].barh(top_movies['title'], top_movies['count'], color='steelblue')
axes[1].set_title('Top 10 Películas más Valoradas', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Número de ratings')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 4. Análisis de Usuarios

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribución por género
gender_counts = users['gender'].value_counts()
axes[0].pie(gender_counts.values, labels=gender_counts.index,
            autopct='%1.1f%%', colors=['steelblue', 'coral'],
            startangle=90)
axes[0].set_title('Distribución por Género', fontsize=14, fontweight='bold')

# Distribución por edad
age_labels = {1: '<18', 18: '18-24', 25: '25-34', 35: '35-44',
              45: '45-49', 50: '50-55', 56: '56+'}
users['age_label'] = users['age'].map(age_labels)
age_counts = users['age_label'].value_counts().reindex(age_labels.values())
axes[1].bar(age_counts.index, age_counts.values, color='mediumpurple', edgecolor='white')
axes[1].set_title('Distribución por Edad', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Grupo de edad')
axes[1].set_ylabel('Número de usuarios')
axes[1].tick_params(axis='x', rotation=30)

# Top 10 ocupaciones
occupation_labels = {
    0: 'Otro', 1: 'Académico', 2: 'Artista', 3: 'Funcionario',
    4: 'Estudiante universitario', 5: 'Técnico', 6: 'Ingeniero',
    7: 'Ejecutivo', 8: 'Autónomo', 9: 'Granjero', 10: 'Ama de casa',
    11: 'Estudiante K-12', 12: 'Abogado', 13: 'Programador',
    14: 'Jubilado', 15: 'Científico', 16: 'Empleado SS',
    17: 'Comercial', 18: 'Artesano', 19: 'Desempleado', 20: 'Escritor'
}
users['occupation_label'] = users['occupation'].map(occupation_labels)
occ_counts = users['occupation_label'].value_counts().head(10)
axes[2].barh(occ_counts.index, occ_counts.values, color='coral')
axes[2].set_title('Top 10 Ocupaciones', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Número de usuarios')
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Rating medio por género de usuario
ratings_users = ratings.merge(users[['userId', 'gender', 'age_label']], on='userId')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rating_by_gender = ratings_users.groupby('gender')['rating'].mean()
axes[0].bar(rating_by_gender.index, rating_by_gender.values,
            color=['steelblue', 'coral'], edgecolor='white')
axes[0].set_title('Rating Medio por Género', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Rating medio')
axes[0].set_ylim(3.4, 3.7)
for i, (idx, val) in enumerate(rating_by_gender.items()):
    axes[0].text(i, val + 0.005, f'{val:.3f}', ha='center', fontweight='bold')

rating_by_age = ratings_users.groupby('age_label')['rating'].mean().reindex(age_labels.values())
axes[1].bar(rating_by_age.index, rating_by_age.values, color='mediumpurple', edgecolor='white')
axes[1].set_title('Rating Medio por Grupo de Edad', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Rating medio')
axes[1].set_ylim(3.4, 3.7)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 5. Análisis de Películas

In [ ]:
# Géneros más frecuentes
all_genres = [genre for genres in movies['genres_parsed'] for genre in genres]
genre_series = pd.Series(all_genres).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(genre_series.index, genre_series.values, color='steelblue')
axes[0].set_title('Géneros más Frecuentes', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Número de películas')
axes[0].invert_yaxis()

# Películas por año
movies_by_year = movies['year'].value_counts().sort_index()
axes[1].plot(movies_by_year.index, movies_by_year.values, color='coral', linewidth=2)
axes[1].fill_between(movies_by_year.index, movies_by_year.values, alpha=0.3, color='coral')
axes[1].set_title('Películas por Año de Lanzamiento', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('Número de películas')

plt.tight_layout()
plt.show()

In [ ]:
# Rating medio por género
ratings_movies = ratings.merge(movies[['movieId', 'genres_parsed']], on='movieId')
ratings_movies = ratings_movies.explode('genres_parsed')
rating_by_genre = ratings_movies.groupby('genres_parsed')['rating'].agg(['mean', 'count'])
rating_by_genre = rating_by_genre[rating_by_genre['count'] > 1000].sort_values('mean', ascending=False)

plt.figure(figsize=(14, 5))
bars = plt.bar(rating_by_genre.index, rating_by_genre['mean'],
               color='mediumpurple', edgecolor='white')
plt.title('Rating Medio por Género (mín. 1000 valoraciones)', fontsize=14, fontweight='bold')
plt.ylabel('Rating medio')
plt.ylim(3.0, 4.2)
plt.xticks(rotation=45, ha='right')
plt.axhline(ratings['rating'].mean(), color='red', linestyle='--',
            label=f'Media global: {ratings["rating"].mean():.2f}')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Análisis de Sparsity (Dispersión de la matriz)

In [ ]:
n_users  = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
n_ratings = len(ratings)

sparsity = 1 - (n_ratings / (n_users * n_movies))

print('=== ANÁLISIS DE SPARSITY ===')
print(f'Usuarios únicos:      {n_users:,}')
print(f'Películas únicas:     {n_movies:,}')
print(f'Ratings totales:      {n_ratings:,}')
print(f'Combinaciones posibles: {n_users * n_movies:,}')
print(f'Sparsity:             {sparsity:.4%}')
print(f'\nSolo el {1-sparsity:.2%} de las combinaciones usuario-película tienen rating.')
print('Esto justifica el uso de técnicas de filtrado colaborativo como SVD,')
print('que pueden inferir ratings en combinaciones no observadas.')

In [ ]:
# Visualización de la sparsity con una muestra
sample_users  = ratings['userId'].unique()[:100]
sample_movies = ratings['movieId'].unique()[:100]

sample = ratings[
    ratings['userId'].isin(sample_users) &
    ratings['movieId'].isin(sample_movies)
]

matrix = sample.pivot_table(index='userId', columns='movieId', values='rating')

plt.figure(figsize=(12, 6))
mask = matrix.isnull()
plt.imshow(~mask.values, cmap='Blues', aspect='auto', interpolation='none')
plt.title('Matriz Usuario-Película (muestra 100x100)\nAzul = tiene rating | Blanco = sin rating',
          fontsize=13, fontweight='bold')
plt.xlabel('Películas')
plt.ylabel('Usuarios')
plt.colorbar(label='Tiene rating')
plt.tight_layout()
plt.show()

## 7. Conclusiones

Del análisis exploratorio se extraen las siguientes conclusiones:

**Ratings:**
- El rating más frecuente es **4**, seguido de **3** y **5**. Los ratings bajos (1 y 2) son minoritarios, lo que indica un sesgo positivo típico en sistemas de recomendación.
- La distribución de ratings por usuario es muy asimétrica: hay usuarios muy activos con cientos de valoraciones y muchos con pocas.

**Usuarios:**
- El dataset está dominado por usuarios masculinos (~72%), con mayor concentración en el rango de edad **25-34 años**.
- Los estudiantes son el grupo de ocupación más grande, lo que es coherente con la naturaleza de la plataforma.

**Películas:**
- El género más común es **Drama**, seguido de **Comedy**. Los géneros mejor valorados tienden a ser **Film-Noir** y **War**, mientras que géneros populares como **Comedy** tienen ratings medios más bajos.
- La producción de películas en el dataset se concentra en la década de los 90, con un pico en 1996-1998.

**Sparsity:**
- La matriz usuario-película tiene una dispersión superior al **95%**, lo que justifica el uso de técnicas como **SVD** que pueden inferir preferencias a partir de patrones latentes en los datos observados.